# Clase 3: continuidad y evidencia verificable

Esta práctica sigue una decisión real desde `model_evaluation` hasta su comprobación: evaluación -> Evidence Record persistido -> payload canónico/digest -> leaf -> árbol Merkle/root -> proof -> batch en Portal -> receipt/lifecycle -> verificación E2E.

## Flujo automático de aula

Ejecuta todas las celdas en orden. N06 selecciona la primera evaluación `USABLE`, reutiliza o crea su Evidence Record y batch, reutiliza o despliega el contrato local cuando Anvil no tenga bytecode y ancla el root. Una segunda ejecución reutiliza esos artefactos y reconcilia el anchor persistido, sin crear duplicados ni reenviar la transacción.

## Empezar limpio

Antes de abrir Jupyter, desde la raíz ejecute `uv run python scripts/reset_demo.py --demo-root var/local-demo` para revisar las rutas. Antes del reset real, cree o reutilice el marcador con `uv run python scripts/bootstrap_local_demo.py --initialize-demo-root`; el reset exige `--confirm-reset-local-demo` y elimina irreversiblemente `invoiceops.db`, sus WAL/SHM, `mlflow.db`, `mlflow-artifacts/`, `notebook-state/state.json` y el dataset canónico `data/invoice-risk-v1` dentro de ese directorio. El reset recrea SQLite; luego inicie MLflow con ese root y ejecute `scripts/bootstrap_local_demo.py --db-path var/local-demo/invoiceops.db` para recrear el dataset aislado, complete 03--05 y copie los IDs visibles. Consulte `notebooks/README.md` para recuperación y comandos completos.

## Inicio

No configures IDs ni flags: el perfil `classroom` contiene el entorno local efímero necesario.


In [ ]:
import os
from html import escape
from pathlib import Path

from IPython.display import HTML, display

from invoiceops.demo_state import inspect_demo_state
from invoiceops.legacy.db import DEFAULT_DB_PATH, PROJECT_ROOT


def full_value(value):
    return f'<details><summary>ver valor completo</summary><code>{escape(str(value))}</code></details>'


def show_table(headers, rows):
    head = ''.join(f'<th>{escape(str(header))}</th>' for header in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))


def precondition(state, next_action, technical=''):
    detail = f'<details><summary>detalle técnico</summary><pre>{escape(technical)}</pre></details>' if technical else ''
    display(HTML(f'<p><strong>Estado:</strong> {escape(state)}<br><strong>Siguiente acción:</strong> {escape(next_action)}</p>{detail}'))


# Match the public INVOICEOPS_DB_PATH configuration: relative paths are rooted at the project.
configured_db_path = Path(os.getenv('INVOICEOPS_DB_PATH', DEFAULT_DB_PATH))
db_path = configured_db_path if configured_db_path.is_absolute() else PROJECT_ROOT / configured_db_path
state = inspect_demo_state().to_dict()
show_table(['Componente', 'Estado', 'Dato'], [
    ['SQLite', escape(state['database']['status']), str(state['database']['model_evaluation_count'])],
    ['Evidence', escape(state['evidence']['status']), escape(str(state['evidence']['usable_evaluation_ids']))],
    ['Anvil', escape(state['evm_runtime']['status']), escape(str(state['evm_runtime']['chain_id']))],
])
print('Lectura solamente: esta celda no modifica SQLite, manifest ni Anvil.')

## 1. Selecciona una evaluación real

La tabla muestra qué evaluaciones tienen lineage suficiente. N06 selecciona automáticamente la primera `USABLE`.

In [ ]:
from invoiceops.evidence import list_evaluation_candidates
from invoiceops.notebook_helpers import evaluation_selection_state

candidates = list_evaluation_candidates(db_path)
show_table(['Evaluación', 'Estado', 'Motivo'], [
    [str(item.evaluation_id), 'USABLE' if item.usable else 'PENDING', escape(item.cause or 'Lineage completo')]
    for item in candidates
])
EVALUATION_ID = next((item.evaluation_id for item in candidates if item.usable), None)
selection = evaluation_selection_state(candidates, EVALUATION_ID)
if selection.ready:
    precondition(f'Evaluación seleccionada automáticamente: ID {EVALUATION_ID}.', 'Continúa.')
else:
    precondition(selection.state, selection.next_action)

## 2. Evidence Record y afirmación trazable

La primera parte es lectura: recupera el Evidence Record ya persistido y muestra la evaluación, factura, modelo, versión, run, política, fuente, razón, correlación y provenance.

N06 reutiliza el Evidence Record si existe; de lo contrario lo crea una vez para la evaluación seleccionada.

In [ ]:
from invoiceops.evidence import (
    EvidenceError,
    build_evidence_record,
    get_evidence_record,
    persist_evidence_records,
)

record = None
if not selection.ready:
    precondition('Sección omitida: falta evaluación.', selection.next_action)
else:
    record = get_evidence_record(db_path, EVALUATION_ID)
    if record is None:
        try:
            record = build_evidence_record(db_path, EVALUATION_ID)
            persist_evidence_records(db_path, [record])
        except EvidenceError as error:
            precondition('No se pudo persistir el Evidence Record.', 'Revisa la evaluación y su lineage; después relee el estado.', repr(error))
    if record is not None:
        show_table(['Campo', 'Valor'], [
            ['Evaluación / fecha', f'{record.evaluation_id} / {escape(record.evaluation_created_at)}'], ['Factura', escape(record.invoice_id)],
            ['Modelo / versión / run', f'{escape(record.model_name)} / {escape(record.model_version)} / {full_value(record.run_id)}'],
            ['Probabilidad / recomendación', f'{escape(record.manual_review_probability)} / {escape(record.recommendation)}'],
            ['Política / umbral', f'{escape(record.policy_version)} / {escape(record.policy_threshold)}'],
            ['Fuente / razón', f'{escape(record.source)} / {escape(record.reason)}'],
            ['Correlation ID', full_value(record.correlation_id)],
            ['Provenance', f'dataset={escape(record.provenance.dataset_version)}; schema={escape(record.provenance.feature_schema_version)}; commit={full_value(record.provenance.git_commit)}'],
        ])

## 3. Payload canónico, bytes y digest

Esta es una lectura en memoria. `canonicalize_evidence_record` fija los bytes UTF-8 y `evidence_digest` calcula el digest completo con la API pública. El JSON completo queda bajo disclosure: los hashes abreviados sirven solo como apoyo visual, nunca como valor de verificación.

In [ ]:
import json

from invoiceops.evidence import canonicalize_evidence_record, evidence_digest

if record is None:
    precondition('Sección omitida: falta Evidence Record.', 'Completa la sección 2 y vuelve a ejecutar esta celda.')
else:
    canonical_payload = canonicalize_evidence_record(record)
    canonical_text = canonical_payload.decode('utf-8')
    canonical_version = json.loads(canonical_text)['canonical_version']
    digest = evidence_digest(record)
    show_table(['Artefacto', 'Valor'], [
        ['Versión canónica', escape(canonical_version)],
        ['Representación', 'UTF-8'],
        ['Cantidad de bytes', str(len(canonical_payload))],
        ['Digest completo', full_value(digest)],
        ['Digest abreviado (solo apoyo)', f'<code>{escape(digest[:12])}...{escape(digest[-6:])}</code>'],
    ])
    display(HTML(f'<details><summary>Mostrar payload canónico completo</summary><pre>{escape(canonical_text)}</pre></details>'))

## 4. Batch verificable

N06 reutiliza el último batch que contiene la evaluación; si no existe, crea el batch canónico de esa evidencia.

In [ ]:
from invoiceops.evidence import (
    create_evidence_batch,
    get_latest_evidence_batch_for_evaluation,
)

persisted_batch = None
proof_item = None
if not selection.ready:
    precondition('Sección omitida: falta evaluación.', selection.next_action)
else:
    persisted_batch = get_latest_evidence_batch_for_evaluation(db_path, EVALUATION_ID)
    if persisted_batch is None:
        persisted_batch = create_evidence_batch(db_path, [EVALUATION_ID])
    proof_item = next(item for item in persisted_batch.items if item.evaluation_id == EVALUATION_ID)
    show_table(['Batch', 'Política', 'Hojas', 'Root completo'], [[str(persisted_batch.id), escape(persisted_batch.policy_version), str(persisted_batch.leaf_count), full_value(persisted_batch.root_hash)]])
    show_table(['Evaluación', 'Índice', 'Leaf completa'], [[str(item.evaluation_id), str(item.leaf_index), full_value(item.leaf_hash)] for item in persisted_batch.items])

## 5. Árbol Merkle: niveles, orden y duplicación

Esta es una lectura en memoria con `merkle_tree`; no recalcula Keccak ni concatena hashes. Cada fila explica el par ordenado `left || right` que la API ya resumió como parent. Cuando falta `right`, la regla pública duplica la última hoja. Los valores completos están bajo disclosure.

In [ ]:
from invoiceops.evidence import merkle_tree

if persisted_batch is None:
    precondition('Sección omitida: falta batch válido.', 'Ejecuta N06 desde el inicio.')
else:
    tree = merkle_tree([item.leaf_hash for item in persisted_batch.items], sort_leaves=False)
    for level_index, level in enumerate(tree.levels):
        rows = []
        if level_index == len(tree.levels) - 1:
            rows.append(['Root', full_value(level.hashes[0]), 'Resultado final de la API pública.'])
        else:
            parents = tree.levels[level_index + 1].hashes
            for pair_index, parent in enumerate(parents):
                left = level.hashes[pair_index * 2]
                right_index = pair_index * 2 + 1
                duplicated = level.duplicates_final_hash and right_index >= len(level.hashes)
                right = left if duplicated else level.hashes[right_index]
                explanation = f'orden: left || right; duplicación impar: {str(duplicated).lower()}'
                values = f'left={full_value(left)}<br>right={full_value(right)}<br>parent={full_value(parent)}'
                rows.append([f'Par {pair_index}', values, explanation])
        show_table([f'Nivel {level_index}', 'Valores completos', 'Regla'], rows)
    show_table(['Comprobación', 'Resultado'], [['Root API / root persistido', str(tree.root == persisted_batch.root_hash).lower()]])

## 6. Proof persistida y alteración en memoria

Esta celda es de solo lectura: valida la proof guardada para la leaf seleccionada y cambia un carácter de una copia en memoria. No escribe SQLite, manifest ni Anvil. Una proof válida demuestra pertenencia al root; la copia alterada debe fallar con la misma proof.

In [ ]:
from invoiceops.evidence import verify_merkle_proof

if proof_item is None or persisted_batch is None:
    precondition('Sección omitida: falta proof persistida.', 'Selecciona un batch válido que incluya la evaluación.')
else:
    proof_valid = verify_merkle_proof(proof_item.leaf_hash, proof_item.proof, persisted_batch.root_hash)
    altered_leaf = ('0' if proof_item.leaf_hash[0] != '0' else '1') + proof_item.leaf_hash[1:]
    altered_valid = verify_merkle_proof(altered_leaf, proof_item.proof, persisted_batch.root_hash)
    show_table(['Caso', 'Leaf completa', 'Resultado'], [
        ['Proof persistida', full_value(proof_item.leaf_hash), str(proof_valid).lower()],
        ['Leaf alterada solo en memoria', full_value(altered_leaf), str(altered_valid).lower()],
    ])
    if not proof_valid or altered_valid:
        precondition('La verificación no produjo el patrón esperado.', 'Relee el batch seleccionado y comprueba que la proof pertenece a esa evaluación.', f'proof_valid={proof_valid}, altered_valid={altered_valid}')

## 7. Contrato local y lifecycle

N06 reutiliza el contrato del manifest si Anvil tiene bytecode en esa dirección. En un Anvil limpio lo despliega una vez; después un anchor `ambiguous` se reconcilia con su receipt persistido sin reenvío.

In [ ]:
import os

from invoiceops.anchor import (
    chain,
    deploy_anchor,
    local_signer,
    resolve_deployment,
)

evm = None
signer = None
deployment = None
if persisted_batch is None:
    precondition('Sección omitida: falta batch válido.', 'Completa las secciones anteriores.')
else:
    evm = chain(os.getenv('INVOICEOPS_EVM_RPC_URL', 'http://127.0.0.1:8545'))
    signer = local_signer(evm)
    deployment = resolve_deployment()
    if not evm.eth.get_code(deployment.address):
        deployment = deploy_anchor(evm, signer, rpc_url=os.getenv('INVOICEOPS_EVM_RPC_URL', 'http://127.0.0.1:8545'))
    show_table(['Precondición', 'Valor'], [['Chain ID', str(evm.eth.chain_id)], ['Signer', full_value(signer)], ['Contrato', escape(deployment.contract)], ['Dirección', full_value(deployment.address)]])

## 8. Despliegue automático

La sección anterior realiza el despliegue sólo si Anvil no tiene bytecode en la dirección del manifest. Esta celda muestra el resultado reutilizado.

In [ ]:
if deployment is None:
    precondition('Sección omitida: falta contrato.', 'Completa la sección anterior.')
else:
    show_table(['Contrato disponible', 'Dirección'], [[escape(deployment.contract), full_value(deployment.address)]])

## 9. Envío y reconciliación automática del anchor

La API reserva el anchor antes de enviar la transacción. En una repetición reutiliza esa reserva y sólo reconcilia el receipt; no crea una segunda transacción.

In [ ]:
from invoiceops.anchor import (
    anchor_evidence_batch,
    inspect_evidence_batch_anchor,
)

if persisted_batch is None or evm is None or deployment is None or signer is None:
    precondition('Sección omitida: faltan batch o preflight.', 'Completa las secciones 4 y 7 antes de modificar un anchor.')
    anchor = None
else:
    anchor = anchor_evidence_batch(db_path, batch_id=persisted_batch.id, root_hash=persisted_batch.root_hash, web3=evm, deployment=deployment, signer=signer)
if anchor is not None:
    persisted_anchor = inspect_evidence_batch_anchor(db_path, anchor.id)
    show_table(['Anchor', 'Lifecycle', 'Tx completa', 'Root completo'], [[str(persisted_anchor.id), escape(persisted_anchor.status), full_value(persisted_anchor.transaction_hash), full_value(persisted_anchor.root_hash)]])

## 10. Verificación E2E

`verify_evidence_batch` recorre los vínculos canónicos, leaf, proof, batch, lifecycle y root on-chain. El resultado final debe ser `PASS`.

In [ ]:
import os

from invoiceops.verification import verify_evidence_batch

if persisted_batch is None or not selection.ready:
    precondition('Sección omitida: faltan precondiciones.', 'Ejecuta N06 desde el inicio.')
else:
    verification = verify_evidence_batch(
        db_path, persisted_batch.id, EVALUATION_ID,
        rpc_url=os.getenv('INVOICEOPS_EVM_RPC_URL', 'http://127.0.0.1:8545'),
    )
    checks = [
        ('Payload canónico -> digest', verification.canonical_hash_valid),
        ('Evidence Record -> leaf', verification.evidence_leaf_valid),
        ('Leaf + proof -> root', verification.proof_valid),
        ('Batch persistido', verification.batch_valid),
        ('Lifecycle persistido', verification.anchor_persisted),
        ('Root on-chain', verification.root_on_chain),
    ]
    show_table(['Check', 'Estado'], [[escape(name), 'PASS' if passed else 'PENDING'] for name, passed in checks])
    if not verification.valid:
        precondition('La verificación E2E está pendiente.', 'Revisa el primer check PENDING y ejecuta N06 desde el inicio.', f'batch_id={persisted_batch.id}, evaluation_id={EVALUATION_ID}')
    else:
        display(HTML('<p><strong>PASS:</strong> los vínculos leídos son consistentes para la evaluación y batch seleccionados.</p>'))